# Stage N-ATLaS models in Google Cloud Storage

Run this notebook in Kaggle to download the pinned GGUF and selected ASR model snapshots from Hugging Face, then upload them once to your GCS bucket. Cloud Run mounts the bucket at `/models`, so files must be stored at the bucket root under `llm/` and `asr/`.

Before running: enable Internet in Kaggle Notebook settings; accept any Hugging Face access terms for the ASR repositories; add Kaggle Secrets named `HF_TOKEN` and `GCP_SERVICE_ACCOUNT_JSON`. The service account JSON must have `roles/storage.objectUser` on the destination bucket. Do not put credentials in notebook cells or outputs. Kaggle storage is temporary, so rerun this notebook if the Kaggle session expires before upload completes.

In [ ]:
%pip -q install --upgrade huggingface_hub google-cloud-storage

In [ ]:
from pathlib import Path
import shutil

BUCKET_NAME = "natlas-bucket"
ASR_LANGUAGES = ["hausa", "igbo", "yoruba"]  # Remove languages you do not need
WORK_DIR = Path("/kaggle/working/natlas-models")

available_bytes = shutil.disk_usage("/kaggle/working").free
print(f"Free Kaggle working storage: {available_bytes / (1024**3):.1f} GiB")
assert BUCKET_NAME != "natlas-bucket", "Set BUCKET_NAME first."
assert set(ASR_LANGUAGES).issubset({"hausa", "igbo", "yoruba"}), "Unsupported ASR language selected."

In [ ]:
import json
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
GCP_SERVICE_ACCOUNT_INFO = json.loads(secrets.get_secret("GCP_SERVICE_ACCOUNT_JSON"))
assert HF_TOKEN, "Add the Hugging Face access token as Kaggle Secret HF_TOKEN."
assert GCP_SERVICE_ACCOUNT_INFO.get("client_email"), "GCP_SERVICE_ACCOUNT_JSON is not a service-account key."
print("Secrets loaded (values hidden).")

## Download pinned model files

The LLM is the Q4_K_M GGUF configured by the backend. Each ASR folder is a complete Hugging Face snapshot, not only the weight file. This cell downloads to Kaggle's temporary working directory; it does not upload anything.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

LLM_REPO = "tosinamuda/N-ATLaS-GGUF"
LLM_REVISION = "af9195261dd163807b31318b9b9d577de5ae3b7e"
LLM_FILENAME = "N-ATLaS-GGUF-Q4_K_M.gguf"
ASR_MODELS = {
    "hausa": ("NCAIR1/Hausa-ASR", "e635b9eda29060c6114c8f4d8b2d903f5c83a44a"),
    "igbo": ("NCAIR1/Igbo-ASR", "180732299d5cba3dc8b289260ac84b7838bb3954"),
    "yoruba": ("NCAIR1/Yoruba-ASR", "d1ae7b8b79c2ccd547d8761effe5057433f3fc7f"),
}

llm_dir = WORK_DIR / "llm"
llm_dir.mkdir(parents=True, exist_ok=True)
llm_file = Path(hf_hub_download(
    repo_id=LLM_REPO,
    filename=LLM_FILENAME,
    revision=LLM_REVISION,
    token=HF_TOKEN,
    local_dir=llm_dir,
))
print(f"LLM downloaded: {llm_file.stat().st_size / (1024**3):.2f} GiB")

for language in ASR_LANGUAGES:
    repo_id, revision = ASR_MODELS[language]
    destination = WORK_DIR / "asr" / language
    snapshot_download(
        repo_id=repo_id,
        revision=revision,
        token=HF_TOKEN,
        local_dir=destination,
    )
    files = [path for path in destination.rglob("*") if path.is_file() and ".cache" not in path.parts]
    total_bytes = sum(path.stat().st_size for path in files)
    print(f"{language}: {len(files)} files, {total_bytes / (1024**3):.2f} GiB")

## Upload to the bucket

Review the bucket name and downloaded file counts first. Set `CONFIRM_UPLOAD = True` in the next cell only when you are ready. Objects are uploaded to the bucket root as `llm/...` and `asr/<language>/...`, matching the `/models` Cloud Run mount.

In [ ]:
CONFIRM_UPLOAD = False  # Change to True to perform the upload
assert CONFIRM_UPLOAD, "Set CONFIRM_UPLOAD = True after reviewing the target bucket."

from google.cloud import storage
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_info(GCP_SERVICE_ACCOUNT_INFO)
storage_client = storage.Client(project=GCP_SERVICE_ACCOUNT_INFO.get("project_id"), credentials=credentials)
bucket = storage_client.bucket(BUCKET_NAME)
assert bucket.exists(), f"Bucket not found or not accessible: {BUCKET_NAME}"

files_to_upload = [
    path for path in WORK_DIR.rglob("*")
    if path.is_file() and ".cache" not in path.parts
]
assert files_to_upload, "No downloaded model files found. Run the download cell first."

uploaded_bytes = 0
for local_file in files_to_upload:
    object_name = local_file.relative_to(WORK_DIR).as_posix()
    blob = bucket.blob(object_name)
    blob.upload_from_filename(str(local_file), timeout=3600)
    uploaded_bytes += local_file.stat().st_size
    print(f"Uploaded {object_name}")

print(f"Uploaded {len(files_to_upload)} files ({uploaded_bytes / (1024**3):.2f} GiB) to gs://{BUCKET_NAME}/")

In [ ]:
expected_objects = ["llm/N-ATLaS-GGUF-Q4_K_M.gguf"]
expected_objects.extend(f"asr/{language}/config.json" for language in ASR_LANGUAGES)
missing_objects = [name for name in expected_objects if not bucket.blob(name).exists()]
if missing_objects:
    raise RuntimeError(f"Upload verification failed for: {missing_objects}")
print("Verified LLM file and selected ASR snapshots in the bucket.")
print("Configure Cloud Run to mount this bucket at /models; use /models/llm/... and /models/asr.")